# Python 的安装和环境设置

前端的运行环境用 Node.js，后端的运行环境用 Python。

## 我们需要 Python 了

运行 Python 代码之前，电脑上必须先安装 Python。先检查当前系统是否已经安装：

Windows 使用：

```powershell
python --version
```

如果能打印出 Python 3.x 的版本号，说明已经有 Python。

安装方式和安装 Node.js 类似：

在 Python 官网下载最新的版本：https://www.python.org/downloads/

## 如果电脑里不止一个 Python

一台电脑上可能有系统自带的 Python、以前安装的软件带来的 Python，以及自己新安装的 Python。要知道命令实际执行的是哪一个，可以查询路径。

Windows：

```powershell
Get-Command python
```

第一条通常显示当前生效的路径；带 -ALL 的命令会列出所有候选路径，排在最前面的通常是当前命令查找到的版本。

如果明确想使用某个版本，可以在 python 后面加上版本号

## venv：让每个项目拥有独立环境

前端用 node_modules/ 隔离项目依赖；Python 的官方方案是给每个项目创建虚拟环境（virtual environment），工具叫 venv。

虚拟环境会记录：

- 当前项目使用的 Python 解释器。
- 当前项目安装的第三方库。
- 这些库的具体版本。

依赖跟着项目走，项目之间就不会因为公共环境的包版本不同而打架。

## 创建、激活和退出虚拟环境

假设项目位于 ~/project_A，并且想使用 Python 3.12：

```bash
cd ~/project_A
python3.12 -m venv .venv
```

执行后会多一个 .venv 虚拟环境目录。创建后还要激活：

macOS / Linux：

```bash
source .venv/bin/activate
```

Windows PowerShell：

```powershell
.venv\Scripts\Activate.ps1
```

激活后，终端提示符前通常会出现 (.venv)。此时：

- python 和 python3 会指向这个项目的 Python。
- pip install 会把库装到当前项目的 .venv 中。
- 关闭终端或执行退出命令后，环境会退出。

退出虚拟环境：

```bash
deactivate
```

## 给虚拟环境一个容易辨认的名字

虚拟环境目录约定使用 .venv，VS Code 等工具能够自动识别它，不建议为了改名而更换目录名。如果同时管理多个项目，可以只改变激活后显示的提示符：

```bash
python3 -m venv --prompt=project_A .venv
```

目录仍然叫 .venv，但激活后提示符会显示 (project_A)。

如果终端每次打开都出现 (base)，通常说明安装并启用了 miniconda 或 Anaconda。conda 也能管理 Python 环境，但它不会像 venv 一样把环境目录放进项目，因此需要自己记住项目和环境的对应关系。

如果希望关闭 conda 的自动激活，可以尝试：

```bash
conda config --set auto_activate false
```

关闭终端再重新打开后，(base) 不会自动出现。conda 没有被卸载，需要使用时仍可以：

```bash
conda activate      # 进入
conda deactivate    # 退出
```

## 在项目中创建后端目录和 .venv

回到贯穿全程的项目：

```bash
cd ~/zero-to-tech
mkdir backend
cd backend
python3 -m venv --prompt=zero-to-tech .venv
```

然后激活：

```bash
source .venv/bin/activate
```

激活后可能看到：

```text
(zero-to-tech) libo@Mac backend %
```

用路径验证当前 Python：

```bash
which python
which python3
```

它们应该都指向类似下面的路径：

```text
/Users/你的用户名/zero-to-tech/backend/.venv/bin/python
/Users/你的用户名/zero-to-tech/backend/.venv/bin/python3
```

Windows 用户使用 .venv\Scripts\Activate.ps1 激活，之后用 where python 检查路径。

前端仍然位于项目根目录，后端位于 backend/，不必为了形式上的对称再创建 frontend/。前后端真正独立的地方是两个进程、两套依赖和两种部署方式，而不是目录是否完全对称。

### 不要把 .venv 提交到 Git

.venv 和前端的 node_modules 一样，都是本地可重新生成的依赖目录，体积较大，不应提交到 Git。

在项目根目录的 .gitignore 中加入：

```gitignore
backend/.venv/
```

.venv 退出后，重新进入 backend/ 干活时要先激活：

```bash
cd ~/zero-to-tech/backend
source .venv/bin/activate
```

VS Code 安装微软官方 Python 扩展后，通常可以自动识别项目中的 .venv。在 VS Code 中新开终端时，它还可以自动激活该环境。

## 用 uv 管理 Python 项目

前面介绍的 `venv` 和 `pip` 已经能完成虚拟环境和第三方库管理。`uv` 是一个更现代的一体化工具，可以把 Python 版本、项目依赖、虚拟环境和命令运行放到同一套工作流里。

### 安装与验证

Windows PowerShell：

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

macOS / Linux：

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

重新打开终端后验证：

```bash
uv --version
```

### 用 uv 初始化项目

在项目根目录中，`uv init` 会创建项目配置文件：

```bash
cd ~/zero-to-tech
uv init
```

会得到
- `pyproject.toml` 它记录项目名称、和直接依赖等。
- `.python-versin` 记录Python 版本约束。

这两个文件不会立即就有
- `uv.lock` 记录解析后的精确版本。
- `.venv/` 是本机实际使用的隔离环境。

如果在某个目录下直接创建项目目录，也可以：

```bash
uv init zero-to-tech --python
```

### 添加依赖并同步环境

使用 `uv add` 添加项目依赖：

```bash
uv add requests
```

`uv add` 会更新 `pyproject.toml`，重新计算 `uv.lock`，并把依赖同步到项目的 `.venv`。

如果已经有项目配置，也可以直接执行：

```bash
uv sync
```

`uv sync` 会根据项目声明和锁文件创建或更新环境。删除 `.venv` 后再次执行它，就可以按锁文件重建环境。

### 用 uv run 运行代码

`uv run` 会自动在当前项目的虚拟环境中运行

```bash
uv run python first_json.py
```

### uv 与 venv、pip 的对应关系

| 传统工具 | uv 工作流 | 作用 |
|------|------|------|
| `python -m venv .venv` | `uv sync` 或 `uv venv` | 创建、同步虚拟环境 |
| `pip install requests` | `uv add requests` | 声明并安装项目依赖 |
| `pip install -r requirements.txt` | `uv sync` | 按项目清单复现环境 |
| 手动运行 `python file.py` | `uv run python file.py` | 在项目环境中运行命令 |

新项目优先使用 `uv add`、`uv sync` 和 `uv run`，因为它们会维护 `pyproject.toml`、`uv.lock` 和 `.venv` 的一致性。`uv pip` 仍然存在，适合兼容已有 pip 或 `requirements.txt` 的项目，但它不会自动替你维护项目声明。

`.venv/` `.uv/` `__pycache__/` 不提交Git。

## 用 Python 运行一次代码

新建文件 ./backend/first_json.py 从课程网页复制代码进去，在项目根目录下执行：

```bash
`uv run python backend\first_json.py`
```

也可以进去 backend 执行：

```bash
`uv run python first_json.py`
```

## uv：安装第三方库的工具

json 是标准库，可直接导入使用。现在安装一个第三方库 requests，它是 Python 中常用的网络请求库，可以理解成“代码版的 curl”。用它访问 5.1 中调用过的 ipify API。

```bash
uv add requests
```

查看安装位置(可以看到在 .venv 里面)：

```bash
uv pip show requests
```

## 用 python 调一次API

backend 下新建 api_demo.py,写入：

```python
import requests

resp = requests.get("http://httpbin.org/ip")
print(resp.json())
```

运行：

```bash
uv run python backend/api_demo.py
```

可以看到返回了公网 IP

## 前端与后端的依赖对应关系

| 前端 | 后端 | 作用 |
|------|------|------|
| npm | uv | 安装第三方包 |
| node_modules/ | .venv/ | 项目本地依赖环境，不提交 |
| package.json | pyproject.toml | 依赖清单，要提交 |